# Algorithm 27: Conservative Denoiser-PPR Lattice Constraint

Faithful PPR-style test for KLDM lattice constraints.

Core idea:

```text
x_t = (F_t, V_t, ell_t)
optimize only ell_t through the denoiser:
    min_delta c_G(D_l(F_t, V_t, ell_t + delta, A, t)) + trust/safety penalties
then denoise x_t*:
    x0_star = D_theta(F_t, V_t, ell_t*, A, t)
then renoise lattice from ell0_star:
    ell_t' ~ q_t(ell | ell0_star)
```

Important: this notebook **does not optimize fractional coordinates or velocity**. `F_t,V_t` remain fixed during projection optimization.


In [1]:
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "platform")
os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")

import importlib
from dataclasses import dataclass, replace
from pathlib import Path
from types import SimpleNamespace
from typing import Any

import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

ROOT = Path.cwd().resolve()
if not ((ROOT / 'configs').exists() and (ROOT / 'src').exists()):
    if ((ROOT.parent / 'configs').exists() and (ROOT.parent / 'src').exists()):
        ROOT = ROOT.parent

import kldmPlus.algorithm25_kldm_pc_cps_lattice as kspace_backend
import kldmPlus.algorithm27_denoiser_ppr_lattice as algo27_backend
kspace_backend = importlib.reload(kspace_backend)
algo27_backend = importlib.reload(algo27_backend)

from kldmPlus.run_sampling_compare import SamplingCompareRunner, set_seed
from kldmPlus.sample_evaluation import build_structure_from_sample, evaluate_csp_reconstruction as evaluate_csp_reconstruction_plus
from kldm.sample_evaluation import evaluate_csp_reconstruction as evaluate_csp_reconstruction_kldm

CONFIG_PATH = ROOT / 'configs' / 'kldm_plus' / 'mp_20' / 'mp20_sampling_compare_em_vs_algorithm10_casal_chart.yaml'
with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)

SAMPLE_SEED = int(os.environ.get('ALGO27_SEED', '2'))
GRAPH_IDS = [int(v.strip()) for v in os.environ.get('ALGO27_GRAPH_IDS', '1,2,3').split(',') if v.strip()]
ALGO27_TOTAL_STEPS = int(os.environ.get('ALGO27_TOTAL_STEPS', '800'))
ALGO27_REMAINING_STEPS = [int(v.strip()) for v in os.environ.get('ALGO27_REMAINING_STEPS', '400').split(',') if v.strip()]
ALGO27_OPT_STEPS = int(os.environ.get('ALGO27_OPT_STEPS', '5'))
ALGO27_OPT_LR = float(os.environ.get('ALGO27_OPT_LR', '0.005'))
ALGO27_RHO_LATTICE = float(os.environ.get('ALGO27_RHO_L', '0.75'))
ALGO27_M_SWEEP = [int(v.strip()) for v in os.environ.get('ALGO27_M_SWEEP', '0,1,2').split(',') if v.strip()]
ALGO27_RUN_STRENGTH_SWEEP = os.environ.get('ALGO27_RUN_STRENGTH_SWEEP', '1').strip().lower() in {'1','true','yes'}
ALGO27_RUN_SHORT_ROLLOUT = os.environ.get('ALGO27_RUN_SHORT_ROLLOUT', '0').strip().lower() in {'1','true','yes'}

runner = SamplingCompareRunner(config_path=CONFIG_PATH)
runner.model.eval()
set_seed(SAMPLE_SEED)

requested_num_targets = max(max(GRAPH_IDS) if GRAPH_IDS else 0, len(GRAPH_IDS), 5)
if int(runner.compare_cfg.get('num_targets', 0)) < requested_num_targets:
    runner.compare_cfg['num_targets'] = int(requested_num_targets)
    runner.compare_cfg['batch_size'] = max(int(runner.compare_cfg.get('batch_size', 0)), int(requested_num_targets))
    runner.loader, runner.lattice_transform = runner._build_loader()

full_batch = next(iter(runner.loader)).to(runner.device)
full_ptr = full_batch.ptr.tolist()
full_num_graphs = max(len(full_ptr) - 1, 0)
available_graph_ids = [int(graph_id) for graph_id in GRAPH_IDS if 1 <= int(graph_id) <= full_num_graphs]
selected_graph_indices0 = [int(graph_id) - 1 for graph_id in available_graph_ids]
selected_items = full_batch.index_select(selected_graph_indices0) if hasattr(full_batch, 'index_select') else full_batch[selected_graph_indices0]
if isinstance(selected_items, list):
    batch = full_batch.__class__.from_data_list(selected_items)
else:
    batch = selected_items
batch = batch.to(runner.device)
ptr = batch.ptr.detach().cpu().tolist()

ALGO27_CACHE: dict[tuple[Any, ...], Any] = {}
result_tables: dict[str, pd.DataFrame] = {}

print(f'algo27 graphs={available_graph_ids} seed={SAMPLE_SEED} total_steps={ALGO27_TOTAL_STEPS} remaining={ALGO27_REMAINING_STEPS} opt_steps={ALGO27_OPT_STEPS} rho={ALGO27_RHO_LATTICE}')


MODELS_PROJECT_ROOT: /workspace/.venv/lib/python3.11/site-packages/mattergen
dataset_cache action=load dataset=mp_20 split=test reason=fresh path=/workspace/data/mp_20/processed/test
dataset_cache action=from_cache_path:start dataset=mp_20 split=test
dataset_cache action=from_cache_path:done dataset=mp_20 split=test
dataset_cache action=builder_build:start dataset=mp_20 split=test
dataset_cache action=builder_build:done dataset=mp_20 split=test
dataset_cache action=load dataset=mp_20 split=test reason=fresh path=/workspace/data/mp_20/processed/test
dataset_cache action=from_cache_path:start dataset=mp_20 split=test
dataset_cache action=from_cache_path:done dataset=mp_20 split=test
dataset_cache action=builder_build:start dataset=mp_20 split=test
dataset_cache action=builder_build:done dataset=mp_20 split=test
algo27 graphs=[1, 2, 3] seed=2 total_steps=800 remaining=[400] opt_steps=5 rho=0.75


In [2]:
@dataclass
class GraphCase:
    graph_id: int
    graph_idx0: int
    atomic_numbers: torch.Tensor
    gt_frac: torch.Tensor
    gt_l_feature: torch.Tensor
    requested_sg: int


def dataframe_result(name: str, rows: list[dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    if 'PASS' not in df.columns:
        df['PASS'] = False
    if 'status' not in df.columns:
        df['status'] = np.where(df['PASS'], 'PASS', 'FAIL')
    result_tables[name] = df
    return df


def safe_display_sorted(df: pd.DataFrame, by: list[str]):
    df = df.copy()
    for key in by:
        if key not in df.columns:
            df[key] = np.nan
    display(df.sort_values(by).reset_index(drop=True))


def safe_metric_float(value) -> float:
    if value is None:
        return float('nan')
    if torch.is_tensor(value):
        value = value.detach().reshape(-1)
        if value.numel() == 0:
            return float('nan')
        value = value[0].item()
    try:
        return float(value)
    except Exception:
        return float('nan')


def safe_metric_bool(value) -> bool:
    if torch.is_tensor(value):
        value = value.detach().reshape(-1)
        if value.numel() == 0:
            return False
        value = value[0].item()
    return bool(value)


def _cache_key(*parts: Any) -> tuple[Any, ...]:
    return tuple(parts)


def _cached(key: tuple[Any, ...], fn):
    if key not in ALGO27_CACHE:
        ALGO27_CACHE[key] = fn()
    return ALGO27_CACHE[key]


def graph_slice(graph_idx0: int) -> tuple[int, int]:
    return int(ptr[graph_idx0]), int(ptr[graph_idx0 + 1])


def graph_tensors(graph_idx0: int) -> dict[str, Any]:
    start, end = graph_slice(graph_idx0)
    return {
        'pos': batch.pos[start:end].detach().clone(),
        'l': batch.l[graph_idx0].detach().clone().reshape(-1),
        'h': batch.atomic_numbers[start:end].detach().clone().to(torch.long),
        'sg': int(torch.as_tensor(batch.space_group).reshape(-1)[graph_idx0].item()),
    }


def load_test_graphs(graph_ids=available_graph_ids) -> list[GraphCase]:
    out = []
    for graph_idx0, graph_id in enumerate(graph_ids):
        g = graph_tensors(graph_idx0)
        out.append(GraphCase(
            graph_id=int(graph_id),
            graph_idx0=int(graph_idx0),
            atomic_numbers=g['h'],
            gt_frac=g['pos'],
            gt_l_feature=g['l'],
            requested_sg=g['sg'],
        ))
    return out


GRAPH_CASES = load_test_graphs(available_graph_ids)
print('loaded_graphs=', [g.graph_id for g in GRAPH_CASES], 'sg=', [g.requested_sg for g in GRAPH_CASES])


def graph_edge_node_index(case: GraphCase) -> torch.Tensor:
    start, end = graph_slice(case.graph_idx0)
    edge_index = batch.edge_node_index
    mask = ((edge_index[0] >= start) & (edge_index[0] < end) & (edge_index[1] >= start) & (edge_index[1] < end))
    return (edge_index[:, mask] - start).detach().clone()


def make_single_graph_batch_view(case: GraphCase, *, ref_tensor: torch.Tensor | None = None):
    device = case.gt_frac.device if ref_tensor is None else ref_tensor.device
    dtype = case.gt_frac.dtype if ref_tensor is None else ref_tensor.dtype
    pos = case.gt_frac.detach().clone().to(device=device, dtype=dtype)
    l = case.gt_l_feature.detach().clone().reshape(1, -1).to(device=device, dtype=case.gt_l_feature.dtype)
    atomic_numbers = case.atomic_numbers.detach().clone().to(device=device, dtype=torch.long)
    batch_index = torch.zeros(pos.shape[0], device=device, dtype=torch.long)
    num_atoms = torch.tensor([int(pos.shape[0])], device=device, dtype=torch.long)
    edge_node_index = graph_edge_node_index(case).to(device=device)
    space_group = torch.tensor([int(case.requested_sg)], device=device, dtype=torch.long)

    class _SingleGraphBatch(SimpleNamespace):
        def to(self, device):
            out = _SingleGraphBatch()
            for key, value in self.__dict__.items():
                setattr(out, key, value.to(device) if torch.is_tensor(value) else value)
            return out

    return _SingleGraphBatch(
        pos=pos,
        l=l,
        atomic_numbers=atomic_numbers,
        batch=batch_index,
        num_atoms=num_atoms,
        num_graphs=1,
        edge_node_index=edge_node_index,
        space_group=space_group,
    )


def remaining_step_to_tau(remaining_step: int) -> float:
    return float(remaining_step) / float(max(ALGO27_TOTAL_STEPS, 1))


loaded_graphs= [1, 2, 3] sg= [227, 4, 194]


In [3]:
def cell_lengths_angles(cell: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    cell = torch.as_tensor(cell).reshape(3, 3)
    lengths = torch.linalg.norm(cell, dim=1)
    def angle(u, v):
        denom = (torch.linalg.norm(u) * torch.linalg.norm(v)).clamp_min(1.0e-12)
        cosine = torch.clamp(torch.dot(u, v) / denom, -1.0, 1.0)
        return torch.rad2deg(torch.acos(cosine))
    return lengths, torch.stack([angle(cell[1], cell[2]), angle(cell[0], cell[2]), angle(cell[0], cell[1])])


def cell_volume(cell: torch.Tensor) -> float:
    return float(torch.abs(torch.det(torch.as_tensor(cell).reshape(3, 3))).detach().item())


def lattice_metric_dict(pred_l: torch.Tensor, target_l: torch.Tensor, case: GraphCase) -> dict[str, float]:
    num_atoms = int(case.atomic_numbers.shape[0])
    pred_cell = kspace_backend.lattice6_to_matrix(pred_l, num_atoms=num_atoms, lattice_transform=runner.lattice_transform)
    gt_cell = kspace_backend.lattice6_to_matrix(target_l, num_atoms=num_atoms, lattice_transform=runner.lattice_transform)
    pred_lengths, pred_angles = cell_lengths_angles(pred_cell)
    gt_lengths, gt_angles = cell_lengths_angles(gt_cell)
    pred_volume = cell_volume(pred_cell)
    gt_volume = cell_volume(gt_cell)
    return {
        'cell_matrix_rmse': float(torch.sqrt(torch.mean((pred_cell - gt_cell) ** 2)).detach().item()),
        'length_rmse': float(torch.sqrt(torch.mean((pred_lengths - gt_lengths) ** 2)).detach().item()),
        'angle_rmse_deg': float(torch.sqrt(torch.mean((pred_angles - gt_angles) ** 2)).detach().item()),
        'volume_abs_error': abs(pred_volume - gt_volume),
        'volume_rel_error': abs(pred_volume - gt_volume) / max(abs(gt_volume), 1.0e-12),
    }


def evaluate_arrays(case: GraphCase, pred_f: torch.Tensor, pred_l: torch.Tensor, pred_h: torch.Tensor) -> dict[str, Any]:
    plus = evaluate_csp_reconstruction_plus(
        pred_f=pred_f,
        pred_l=pred_l,
        pred_a=pred_h.to(torch.long),
        target_f=case.gt_frac,
        target_l=case.gt_l_feature,
        target_a=case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
        requested_space_group=case.requested_sg,
        validity_cutoff=0.5,
    )
    kldm = evaluate_csp_reconstruction_kldm(
        pred_f=pred_f,
        pred_l=pred_l,
        pred_a=pred_h.to(torch.long),
        target_f=case.gt_frac,
        target_l=case.gt_l_feature,
        target_a=case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
    )
    return {'match': bool(kldm.match), 'valid': bool(kldm.valid), 'rmse': kldm.rmse, 'frac_rmse': plus.frac_rmse, 'plus_match': bool(plus.match), 'plus_valid': bool(plus.valid), 'plus_rmse': plus.rmse}


def detect_spacegroup_family(pred_f: torch.Tensor, pred_l: torch.Tensor, pred_a: torch.Tensor) -> dict[str, Any]:
    try:
        structure = build_structure_from_sample(pred_f, pred_l, pred_a, lattice_transform=runner.lattice_transform)
        sg = int(SpacegroupAnalyzer(structure, symprec=1.0e-1, angle_tolerance=5.0).get_space_group_number())
        return {'detected_space_group': sg, 'detected_family': kspace_backend.spacegroup_to_crystal_family(sg)}
    except Exception:
        return {'detected_space_group': None, 'detected_family': None}


def algo27_config(**overrides) -> algo27_backend.Algorithm27Config:
    cfg = algo27_backend.Algorithm27Config(
        optimizer_steps=int(ALGO27_OPT_STEPS),
        optimizer_lr=float(ALGO27_OPT_LR),
        lambda_k=1.0,
        lambda_trust=2.0,
        lambda_cart=2.0,
        lambda_volume=0.05,
        rho_lattice=float(ALGO27_RHO_LATTICE),
        max_delta_norm=0.20,
        max_induced_cart_shift=0.10,
        max_relative_cell_shift=0.05,
        max_angle_change_deg=3.0,
        use_safety_gate=True,
    )
    return replace(cfg, **overrides)


In [4]:
def local_noisy_state(case: GraphCase, *, remaining_step: int, seed: int = SAMPLE_SEED) -> dict[str, Any]:
    tau = remaining_step_to_tau(remaining_step)
    batch_view = make_single_graph_batch_view(case, ref_tensor=case.gt_frac)
    set_seed(int(seed) + 1000 * int(case.graph_id) + int(remaining_step))
    t_graph = torch.as_tensor([[tau]], device=case.gt_frac.device, dtype=case.gt_frac.dtype)
    t_nodes = torch.full((int(case.gt_frac.shape[0]),), tau, device=case.gt_frac.device, dtype=case.gt_frac.dtype)
    f_t, v_t, *_ = runner.model.tdm.sample_noisy_state(t=t_nodes, f0=case.gt_frac, index=batch_view.batch)
    t_lattice = t_graph.reshape(-1)
    set_seed(int(seed) + 2000 * int(case.graph_id) + int(remaining_step))
    l_t, _ = runner.model.diffusion_l.forward_sample(
        t=t_lattice,
        x0=case.gt_l_feature.reshape(1, -1),
        num_atoms=batch_view.num_atoms,
    )
    return {
        'f_t': f_t,
        'v_t': v_t,
        'l_t': l_t.reshape(1, -1),
        'a_t': case.atomic_numbers,
        'node_index': batch_view.batch,
        'edge_node_index': batch_view.edge_node_index,
        't_graph': t_graph,
        't_nodes': t_nodes,
        't_lattice': t_lattice,
        'batch_view': batch_view,
        'tau': tau,
    }


def clean_prediction(case: GraphCase, state: dict[str, Any]) -> algo27_backend.Algorithm27CleanPrediction:
    return algo27_backend.predict_clean_state(
        model=runner.model,
        f_t=state['f_t'],
        v_t=state['v_t'],
        l_t=state['l_t'],
        atom_types=state['a_t'],
        node_index=state['node_index'],
        edge_node_index=state['edge_node_index'],
        t_graph=state['t_graph'],
        t_nodes=state['t_nodes'],
        t_lattice=state['t_lattice'],
        num_atoms=state['batch_view'].num_atoms,
        detach=True,
    )


def run_branch(case: GraphCase, state: dict[str, Any], mode: str, cfg: algo27_backend.Algorithm27Config) -> algo27_backend.Algorithm27PPRResult:
    return algo27_backend.run_algorithm27_branch(
        mode=mode,
        model=runner.model,
        f_t=state['f_t'],
        v_t=state['v_t'],
        l_t=state['l_t'],
        atom_types=state['a_t'],
        node_index=state['node_index'],
        edge_node_index=state['edge_node_index'],
        t_graph=state['t_graph'],
        t_nodes=state['t_nodes'],
        t_lattice=state['t_lattice'],
        num_atoms=state['batch_view'].num_atoms,
        lattice_transform=runner.lattice_transform,
        space_group_number=int(case.requested_sg),
        tau=float(state['tau']),
        config=cfg,
    )


def branch_row(case: GraphCase, state: dict[str, Any], remaining_step: int, result: algo27_backend.Algorithm27PPRResult, cfg: algo27_backend.Algorithm27Config, test: str) -> dict[str, Any]:
    before_lat = lattice_metric_dict(result.clean_before.ell0_hat, case.gt_l_feature, case)
    star_lat = lattice_metric_dict(result.clean_star.ell0_hat, case.gt_l_feature, case)
    after_clean = clean_prediction(case, {**state, 'l_t': result.l_t.reshape(1, -1)})
    after_lat = lattice_metric_dict(after_clean.ell0_hat, case.gt_l_feature, case)
    d = result.diagnostics
    return {
        'test': test,
        'graph': case.graph_id,
        'space_group': int(case.requested_sg),
        'remaining_step': int(remaining_step),
        'tau': remaining_step_to_tau(int(remaining_step)),
        'mode': result.mode,
        'notes': result.notes,
        'optimizer_steps': int(cfg.optimizer_steps),
        'optimizer_lr': float(cfg.optimizer_lr),
        'rho_lattice': float(cfg.rho_lattice),
        'lambda_k': float(cfg.lambda_k),
        'lambda_trust': float(cfg.lambda_trust),
        'lambda_cart': float(cfg.lambda_cart),
        'initial_violation': safe_metric_float(d.initial_violation),
        'final_violation': safe_metric_float(d.final_violation),
        'violation_reduction': safe_metric_float(d.initial_violation - d.final_violation),
        'accepted_by_safety': safe_metric_bool(d.accepted),
        'clipped': safe_metric_bool(d.clipped),
        'induced_cart_shift_rms': safe_metric_float(d.induced_cart_shift_rms),
        'relative_cell_shift': safe_metric_float(d.relative_cell_shift),
        'max_angle_change_deg': safe_metric_float(d.max_angle_change_deg),
        'noisy_lattice_shift_norm': safe_metric_float(d.noisy_lattice_shift_norm),
        'clean_lattice_rmse_before': before_lat['cell_matrix_rmse'],
        'clean_lattice_rmse_star': star_lat['cell_matrix_rmse'],
        'clean_lattice_rmse_after_branch': after_lat['cell_matrix_rmse'],
        'length_rmse_after_branch': after_lat['length_rmse'],
        'angle_rmse_after_branch_deg': after_lat['angle_rmse_deg'],
        'PASS': True,
        'status': 'INFO',
    }


### Test 0: Objective Sensitivity

Before asking the optimizer to solve anything, check whether the PPR objective has a usable local signal.

This tests:

```text
grad_l c_G(D_l(F,V,l_t,A,t))
finite-difference best improvement over single lattice-feature nudges
whether k-space fell onto the constant invalid-penalty plateau
```

If `grad_norm` and `best_fd_reduction` are both near zero, Algorithm27 cannot work at this step/objective no matter how long we optimize.


In [ ]:
rows_t0 = []
for case in GRAPH_CASES:
    for remaining_step in ALGO27_REMAINING_STEPS[:1]:
        state = local_noisy_state(case, remaining_step=int(remaining_step))
        cfg = algo27_config()
        sens = algo27_backend.lattice_objective_sensitivity(
            model=runner.model,
            f_t=state['f_t'],
            v_t=state['v_t'],
            l_t=state['l_t'],
            atom_types=state['a_t'],
            node_index=state['node_index'],
            edge_node_index=state['edge_node_index'],
            t_graph=state['t_graph'],
            t_nodes=state['t_nodes'],
            t_lattice=state['t_lattice'],
            num_atoms=state['batch_view'].num_atoms,
            lattice_transform=runner.lattice_transform,
            space_group_number=int(case.requested_sg),
            config=cfg,
        )
        rows_t0.append({
            'test': 'algorithm27_t0_objective_sensitivity',
            'graph': case.graph_id,
            'space_group': int(case.requested_sg),
            'remaining_step': int(remaining_step),
            'tau': remaining_step_to_tau(int(remaining_step)),
            'initial_violation': safe_metric_float(sens.initial_violation),
            'grad_norm': safe_metric_float(sens.grad_norm),
            'max_abs_grad': safe_metric_float(sens.max_abs_grad),
            'best_fd_violation': safe_metric_float(sens.best_fd_violation),
            'best_fd_coord': int(sens.best_fd_coord),
            'best_fd_delta': safe_metric_float(sens.best_fd_delta),
            'best_fd_reduction': safe_metric_float(sens.best_fd_reduction),
            'invalid_penalty_plateau': bool(sens.invalid_penalty_plateau),
            'objective_has_signal': bool((sens.grad_norm > 1e-10) or (sens.best_fd_reduction > 1e-8)),
            'PASS': bool((sens.grad_norm > 1e-10) or (sens.best_fd_reduction > 1e-8)),
            'status': 'INFO',
        })
safe_display_sorted(dataframe_result('algorithm27_t0_objective_sensitivity', rows_t0), ['graph', 'remaining_step'])


### Test 1: Optimizer Sanity

For graphs 1–3 at selected `t`, optimize only `ell_t` through the denoiser.

Success means:

```text
violation decreases
safety accepts or clearly rejects
clean lattice RMSE does not explode
```


In [5]:
rows_t1 = []
for case in GRAPH_CASES:
    for remaining_step in ALGO27_REMAINING_STEPS:
        state = local_noisy_state(case, remaining_step=int(remaining_step))
        cfg = algo27_config()
        result = run_branch(case, state, 'denoiser_ppr', cfg)
        rows_t1.append(branch_row(case, state, int(remaining_step), result, cfg, 'algorithm27_t1_optimizer_sanity'))

safe_display_sorted(dataframe_result('algorithm27_t1_optimizer_sanity', rows_t1), ['graph', 'remaining_step'])


,test,graph,space_group,remaining_step,tau,mode,notes,optimizer_steps,optimizer_lr,rho_lattice,...,relative_cell_shift,max_angle_change_deg,noisy_lattice_shift_norm,clean_lattice_rmse_before,clean_lattice_rmse_star,clean_lattice_rmse_after_branch,length_rmse_after_branch,angle_rmse_after_branch_deg,PASS,status
0,algorithm27_t1_optimizer_sanity,1,227,400,0.5,denoiser_ppr,optimize_through_denoiser_then_lattice_renoise,5,0.005,0.75,...,0.0,0.0,0.0,0.177237,0.177237,0.263840,0.395116,2.463607,True,INFO
1,algorithm27_t1_optimizer_sanity,2,4,400,0.5,denoiser_ppr,optimize_through_denoiser_then_lattice_renoise,5,0.005,0.75,...,0.0,0.0,0.0,0.166267,0.166267,0.216680,0.215492,2.763315,True,INFO
2,algorithm27_t1_optimizer_sanity,3,194,400,0.5,denoiser_ppr,optimize_through_denoiser_then_lattice_renoise,5,0.005,0.75,...,0.0,0.0,0.0,0.340573,0.340573,0.401698,0.438515,5.511470,True,INFO


### Test 1b: Optimizer Stress Test Without Safety

This deliberately removes trust/cart/safety pressure and uses a larger LR/radius. It is **not** a sampler setting.

Purpose: distinguish “optimizer/objective cannot move” from “our conservative safety settings froze a useful optimizer.”


In [ ]:
rows_t1b = []
for case in GRAPH_CASES:
    for remaining_step in ALGO27_REMAINING_STEPS[:1]:
        state = local_noisy_state(case, remaining_step=int(remaining_step))
        cfg = algo27_config(
            optimizer_steps=max(10, ALGO27_OPT_STEPS * 2),
            optimizer_lr=max(0.03, ALGO27_OPT_LR * 10),
            lambda_k=1.0,
            lambda_trust=0.0,
            lambda_cart=0.0,
            lambda_volume=0.0,
            rho_lattice=1.0,
            max_delta_norm=1.0,
            use_safety_gate=False,
            early_stop_relative_improvement=0.99,
        )
        result = run_branch(case, state, 'denoiser_no_renoise', cfg)
        row = branch_row(case, state, int(remaining_step), result, cfg, 'algorithm27_t1b_optimizer_stress_no_safety')
        row['stress_test'] = 'no_trust_no_cart_no_safety'
        row['optimizer_found_direction'] = bool(row['noisy_lattice_shift_norm'] > 1e-8 and row['violation_reduction'] > 1e-10)
        rows_t1b.append(row)
safe_display_sorted(dataframe_result('algorithm27_t1b_optimizer_stress_no_safety', rows_t1b), ['graph', 'remaining_step'])


### Test 2: PPR Component Ablation

Compare the mechanisms requested by the pipeline:

```text
baseline
renoise_no_projection
clean_projection_ppr
cps
denoiser_no_renoise
denoiser_ppr
```

This asks whether projection-through-denoiser and renoising each matter.

In [6]:
rows_t2 = []
MODES_T2 = ['baseline', 'renoise_no_projection', 'clean_projection_ppr', 'cps', 'denoiser_no_renoise', 'denoiser_ppr']
for case in GRAPH_CASES:
    for remaining_step in ALGO27_REMAINING_STEPS[:1]:
        state = local_noisy_state(case, remaining_step=int(remaining_step))
        cfg = algo27_config()
        for mode in MODES_T2:
            set_seed(SAMPLE_SEED + 10_000 * case.graph_id + 100 * int(remaining_step) + MODES_T2.index(mode))
            result = run_branch(case, state, mode, cfg)
            rows_t2.append(branch_row(case, state, int(remaining_step), result, cfg, 'algorithm27_t2_component_ablation'))

safe_display_sorted(dataframe_result('algorithm27_t2_component_ablation', rows_t2), ['graph', 'remaining_step', 'mode'])


,test,graph,space_group,remaining_step,tau,mode,notes,optimizer_steps,optimizer_lr,rho_lattice,...,relative_cell_shift,max_angle_change_deg,noisy_lattice_shift_norm,clean_lattice_rmse_before,clean_lattice_rmse_star,clean_lattice_rmse_after_branch,length_rmse_after_branch,angle_rmse_after_branch_deg,PASS,status
0,algorithm27_t2_component_ablation,1,227,400,0.5,baseline,no_projection_no_renoise,5,0.005,0.75,...,0.0,0.0,0.0,0.177237,0.177237,0.177237,0.263586,1.671322,True,INFO
1,algorithm27_t2_component_ablation,1,227,400,0.5,clean_projection_ppr,direct_clean_kspace_projection_then_renoise,5,0.005,0.75,...,0.0,0.0,0.0,0.177237,1.380680,0.198801,0.324983,1.346434,True,INFO
2,algorithm27_t2_component_ablation,1,227,400,0.5,cps,cps_clean_projection_state_update,5,0.005,0.75,...,0.0,0.0,0.0,0.177237,0.187858,0.177478,0.263856,1.673141,True,INFO
3,algorithm27_t2_component_ablation,1,227,400,0.5,denoiser_no_renoise,optimized_noisy_lattice_without_renoise,5,0.005,0.75,...,0.0,0.0,0.0,0.177237,0.177237,0.177237,0.263586,1.671322,True,INFO
4,algorithm27_t2_component_ablation,1,227,400,0.5,denoiser_ppr,optimize_through_denoiser_then_lattice_renoise,5,0.005,0.75,...,0.0,0.0,0.0,0.177237,0.177237,0.242294,0.333105,2.936726,True,INFO
5,algorithm27_t2_component_ablation,1,227,400,0.5,renoise_no_projection,renoise_from_original_clean_estimate,5,0.005,0.75,...,0.0,0.0,0.0,0.177237,0.177237,0.271912,0.398456,2.868586,True,INFO
6,algorithm27_t2_component_ablation,2,4,400,0.5,baseline,no_projection_no_renoise,5,0.005,0.75,...,0.0,0.0,0.0,0.166267,0.166267,0.166267,0.169709,2.281330,True,INFO
7,algorithm27_t2_component_ablation,2,4,400,0.5,clean_projection_ppr,direct_clean_kspace_projection_then_renoise,5,0.005,0.75,...,0.0,0.0,0.0,0.166267,0.130800,0.159708,0.091901,2.303186,True,INFO
8,algorithm27_t2_component_ablation,2,4,400,0.5,cps,cps_clean_projection_state_update,5,0.005,0.75,...,0.0,0.0,0.0,0.166267,0.163296,0.166316,0.169748,2.282212,True,INFO
9,algorithm27_t2_component_ablation,2,4,400,0.5,denoiser_no_renoise,optimized_noisy_lattice_without_renoise,5,0.005,0.75,...,0.0,0.0,0.0,0.166267,0.166267,0.166267,0.169709,2.281330,True,INFO


### Test 3: Repeated PPR Kernel Sweep

PPR predicts that repeated project-renoise kernels should improve constraint satisfaction, but may cost RMSE/stability.

```text
M=0: baseline
M=1: one denoiser-PPR kernel
M=2: two repeated kernels
```


In [7]:
rows_t3 = []
for case in GRAPH_CASES:
    for remaining_step in ALGO27_REMAINING_STEPS[:1]:
        state = local_noisy_state(case, remaining_step=int(remaining_step))
        cfg = algo27_config()
        baseline = run_branch(case, state, 'baseline', cfg)
        rows_t3.append({**branch_row(case, state, int(remaining_step), baseline, cfg, 'algorithm27_t3_m_sweep'), 'M': 0})
        for m in [v for v in ALGO27_M_SWEEP if int(v) > 0]:
            l_final, results = algo27_backend.repeat_algorithm27_kernel(
                model=runner.model,
                f_t=state['f_t'],
                v_t=state['v_t'],
                l_t=state['l_t'],
                atom_types=state['a_t'],
                node_index=state['node_index'],
                edge_node_index=state['edge_node_index'],
                t_graph=state['t_graph'],
                t_nodes=state['t_nodes'],
                t_lattice=state['t_lattice'],
                num_atoms=state['batch_view'].num_atoms,
                lattice_transform=runner.lattice_transform,
                space_group_number=int(case.requested_sg),
                tau=float(state['tau']),
                repeats=int(m),
                config=cfg,
            )
            result = results[-1]
            rows_t3.append({**branch_row(case, state, int(remaining_step), result, cfg, 'algorithm27_t3_m_sweep'), 'M': int(m), 'num_kernels': len(results)})

safe_display_sorted(dataframe_result('algorithm27_t3_m_sweep', rows_t3), ['graph', 'remaining_step', 'M'])


,test,graph,space_group,remaining_step,tau,mode,notes,optimizer_steps,optimizer_lr,rho_lattice,...,noisy_lattice_shift_norm,clean_lattice_rmse_before,clean_lattice_rmse_star,clean_lattice_rmse_after_branch,length_rmse_after_branch,angle_rmse_after_branch_deg,PASS,status,M,num_kernels
0,algorithm27_t3_m_sweep,1,227,400,0.5,baseline,no_projection_no_renoise,5,0.005,0.75,...,0.0,0.177237,0.177237,0.177237,0.263586,1.671322,True,INFO,0,NaN
1,algorithm27_t3_m_sweep,1,227,400,0.5,denoiser_ppr,optimize_through_denoiser_then_lattice_renoise,5,0.005,0.75,...,0.0,0.177237,0.177237,0.263840,0.395116,2.463607,True,INFO,1,1.0
2,algorithm27_t3_m_sweep,1,227,400,0.5,denoiser_ppr,optimize_through_denoiser_then_lattice_renoise,5,0.005,0.75,...,0.0,0.212608,0.212608,0.247589,0.354545,2.546393,True,INFO,2,2.0
3,algorithm27_t3_m_sweep,2,4,400,0.5,baseline,no_projection_no_renoise,5,0.005,0.75,...,0.0,0.166267,0.166267,0.166267,0.169709,2.281330,True,INFO,0,NaN
4,algorithm27_t3_m_sweep,2,4,400,0.5,denoiser_ppr,optimize_through_denoiser_then_lattice_renoise,5,0.005,0.75,...,0.0,0.166267,0.166267,0.216680,0.215492,2.763315,True,INFO,1,1.0
5,algorithm27_t3_m_sweep,2,4,400,0.5,denoiser_ppr,optimize_through_denoiser_then_lattice_renoise,5,0.005,0.75,...,0.0,0.167756,0.167756,0.158141,0.173087,1.906976,True,INFO,2,2.0
6,algorithm27_t3_m_sweep,3,194,400,0.5,baseline,no_projection_no_renoise,5,0.005,0.75,...,0.0,0.340573,0.340573,0.340573,0.369385,4.399380,True,INFO,0,NaN
7,algorithm27_t3_m_sweep,3,194,400,0.5,denoiser_ppr,optimize_through_denoiser_then_lattice_renoise,5,0.005,0.75,...,0.0,0.340573,0.340573,0.401698,0.438515,5.511470,True,INFO,1,1.0
8,algorithm27_t3_m_sweep,3,194,400,0.5,denoiser_ppr,optimize_through_denoiser_then_lattice_renoise,5,0.005,0.75,...,0.0,0.391537,0.391537,0.334348,0.288830,4.789931,True,INFO,2,2.0


### Test 4: Conservative Strength Sweep

Small sweep over optimization strength on graph 1 only by default. This keeps the notebook lightweight while checking whether the optimizer is too weak or too aggressive.

In [8]:
rows_t4 = []
if ALGO27_RUN_STRENGTH_SWEEP and GRAPH_CASES:
    case = GRAPH_CASES[0]
    remaining_step = int(ALGO27_REMAINING_STEPS[0])
    state = local_noisy_state(case, remaining_step=remaining_step)
    configs = [
        ('weak', algo27_config(lambda_k=0.5, lambda_trust=4.0, lambda_cart=4.0, optimizer_steps=max(3, ALGO27_OPT_STEPS))),
        ('default', algo27_config()),
        ('strong', algo27_config(lambda_k=2.0, lambda_trust=1.0, lambda_cart=2.0, optimizer_steps=max(5, ALGO27_OPT_STEPS))),
    ]
    for label, cfg in configs:
        result = run_branch(case, state, 'denoiser_ppr', cfg)
        rows_t4.append({**branch_row(case, state, remaining_step, result, cfg, 'algorithm27_t4_strength_sweep'), 'strength': label})
else:
    print('Strength sweep disabled. Set ALGO27_RUN_STRENGTH_SWEEP=1 to enable.', flush=True)

safe_display_sorted(dataframe_result('algorithm27_t4_strength_sweep', rows_t4), ['graph', 'strength'])


,test,graph,space_group,remaining_step,tau,mode,notes,optimizer_steps,optimizer_lr,rho_lattice,...,max_angle_change_deg,noisy_lattice_shift_norm,clean_lattice_rmse_before,clean_lattice_rmse_star,clean_lattice_rmse_after_branch,length_rmse_after_branch,angle_rmse_after_branch_deg,PASS,status,strength
0,algorithm27_t4_strength_sweep,1,227,400,0.5,denoiser_ppr,optimize_through_denoiser_then_lattice_renoise,5,0.005,0.75,...,0.0,0.0,0.177237,0.177237,0.212608,0.335158,1.744282,True,INFO,default
1,algorithm27_t4_strength_sweep,1,227,400,0.5,denoiser_ppr,optimize_through_denoiser_then_lattice_renoise,5,0.005,0.75,...,0.0,0.0,0.177237,0.177237,0.199698,0.263512,2.387169,True,INFO,strong
2,algorithm27_t4_strength_sweep,1,227,400,0.5,denoiser_ppr,optimize_through_denoiser_then_lattice_renoise,5,0.005,0.75,...,0.0,0.0,0.177237,0.177237,0.263840,0.395116,2.463607,True,INFO,weak


### Optional Test 5: Short Rollout Placeholder

Disabled by default. Once local optimizer/PPR metrics are positive, add a 50–100 step continuation from selected branches.

In [9]:
rows_t5 = []
if ALGO27_RUN_SHORT_ROLLOUT:
    print('Short rollout is intentionally not wired yet; first inspect Tests 1-4.', flush=True)
else:
    print('Short rollout disabled. Set ALGO27_RUN_SHORT_ROLLOUT=1 after local tests pass.', flush=True)

safe_display_sorted(dataframe_result('algorithm27_t5_short_rollout_placeholder', rows_t5), ['graph'])


Short rollout disabled. Set ALGO27_RUN_SHORT_ROLLOUT=1 after local tests pass.


,PASS,status,graph


### Verdict

This is not a full sampler claim. It only decides whether denoiser-through-PPR is locally worth a short rollout.

In [ ]:
rows_v = []
t0 = result_tables.get('algorithm27_t0_objective_sensitivity', pd.DataFrame())
t1 = result_tables.get('algorithm27_t1_optimizer_sanity', pd.DataFrame())
t2 = result_tables.get('algorithm27_t2_component_ablation', pd.DataFrame())
t3 = result_tables.get('algorithm27_t3_m_sweep', pd.DataFrame())
objective_has_signal = bool(len(t0) and t0['objective_has_signal'].any())
invalid_plateau = bool(len(t0) and t0['invalid_penalty_plateau'].any())
optimizer_moved = bool(len(t1) and (t1['noisy_lattice_shift_norm'] > 1e-8).any())
optimizer_reduces = bool(len(t1) and (t1['violation_reduction'] > 1e-10).any())
safety_accepts_useful = bool(len(t1) and ((t1['accepted_by_safety']) & (t1['noisy_lattice_shift_norm'] > 1e-8)).any())
denoiser_ppr_rows = t2[t2['mode'] == 'denoiser_ppr'] if len(t2) else pd.DataFrame()
clean_proj_rows = t2[t2['mode'] == 'clean_projection_ppr'] if len(t2) else pd.DataFrame()
component_promising = False
if len(denoiser_ppr_rows) and len(clean_proj_rows):
    merged = denoiser_ppr_rows.merge(clean_proj_rows, on=['graph', 'remaining_step'], suffixes=('_denoiser', '_cleanproj'))
    component_promising = bool(len(merged) and (merged['clean_lattice_rmse_after_branch_denoiser'] <= merged['clean_lattice_rmse_after_branch_cleanproj']).any())
m_sweep_improves = bool(len(t3) and (t3.groupby('graph')['final_violation'].min() < t3.groupby('graph')['initial_violation'].first()).any())
if objective_has_signal and optimizer_moved and optimizer_reduces and (component_promising or m_sweep_improves):
    recommendation = 'algorithm27_locally_promising_try_short_rollout'
elif not objective_has_signal:
    recommendation = 'objective_has_no_signal_debug_gradient_or_basis'
elif not optimizer_moved:
    recommendation = 'optimizer_has_signal_but_is_frozen_tune_lr_trust_safety'
elif optimizer_reduces:
    recommendation = 'optimizer_reduces_constraint_but_branch_selection_safety_needs_tuning'
else:
    recommendation = 'do_not_run_sampler_yet_fix_objective_or_basis'
rows_v.append({
    'test': 'algorithm27_verdict',
    'objective_has_signal': objective_has_signal,
    'invalid_penalty_plateau_seen': invalid_plateau,
    'optimizer_moved_lattice': optimizer_moved,
    'optimizer_reduces_violation': optimizer_reduces,
    'safety_accepts_useful_move': safety_accepts_useful,
    'component_promising_vs_clean_projection': component_promising,
    'm_sweep_improves_violation': m_sweep_improves,
    'recommendation': recommendation,
    'PASS': bool(objective_has_signal and optimizer_moved and optimizer_reduces),
    'status': 'INFO',
})
safe_display_sorted(dataframe_result('algorithm27_verdict', rows_v), ['test'])
